# 09 Style LLM Inference

Load a trained Style LLM checkpoint and generate stylized rewrites.
For each author, prints: draft (input) → generated rewrite → real passage.

In [ ]:
import os
import random
import sys
from pathlib import Path

def find_project_root():
    env_root = os.getenv("VOICE_CLONE_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "voice_clone").exists():
            return candidate
    raise RuntimeError("Could not find project root. Set VOICE_CLONE_ROOT to the repo path.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

import torch
from transformers import AutoTokenizer, BitsAndBytesConfig

from voice_clone.data.io import read_jsonl
from voice_clone.data.dataset import triplet_from_row
from voice_clone.models import StyleConditionedCausalLM

print("ROOT:", ROOT)

In [ ]:
CHECKPOINT_PATH = ROOT / "checkpoints" / "style_llm_stage1.pt"
AUTHOR_EMBEDDINGS_PATH = ROOT / "data" / "processed" / "author_style_embeddings.pt"
TRIPLETS_PATH = ROOT / "data" / "processed" / "triplets_style_regen_large.jsonl"

SEED = 42
SAMPLES_PER_AUTHOR = 2   # how many passages to test per author
MAX_NEW_TOKENS = 350
DRAFT_CHARS = 1200
PRINT_CHARS = 600        # chars to print for each section
TEMPERATURE = 0.8
TOP_P = 0.9

random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)
print("checkpoint:", CHECKPOINT_PATH)

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
base_model_name = checkpoint["base_model"]
print("base model:", base_model_name)
print("trained at step:", checkpoint.get("step"))

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = StyleConditionedCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=bnb_config,
)
model.style_projection.load_state_dict(checkpoint["style_projection_state_dict"])
model.style_projection = model.style_projection.to(device)
model.eval()
print("model loaded.")

In [ ]:
author_payload = torch.load(AUTHOR_EMBEDDINGS_PATH, map_location="cpu")
author_embeddings = author_payload["author_embeddings"]

records = [triplet_from_row(row) for row in read_jsonl(TRIPLETS_PATH)]
records = [r for r in records if r.author_id in author_embeddings and (r.style_regenerated_draft or r.neutral_draft)]

from collections import defaultdict
by_author = defaultdict(list)
for r in records:
    by_author[r.author_id].append(r)

print("authors:", sorted(by_author.keys()))
print("records per author:", {a: len(v) for a, v in sorted(by_author.items())})

In [ ]:
def clip_text(text, chars):
    return " ".join(text[:chars].split())


def make_prompt(draft):
    return (
        "Rewrite the draft below so it matches the target author's style while preserving meaning.\n\n"
        "DRAFT:\n"
        f"{draft}\n\n"
        "REWRITE:\n"
    )


def generate_stylized(record, max_new_tokens=MAX_NEW_TOKENS):
    draft = clip_text(record.style_regenerated_draft or record.neutral_draft, DRAFT_CHARS)
    prompt = make_prompt(draft)

    style_emb = author_embeddings[record.author_id].unsqueeze(0).to(device)
    with torch.no_grad():
        model._style_shift = model.style_projection(style_emb)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
        )

    model._style_shift = None
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), draft

In [ ]:
rng = random.Random(SEED)

for author_id in sorted(by_author.keys()):
    pool = by_author[author_id]
    samples = rng.sample(pool, min(SAMPLES_PER_AUTHOR, len(pool)))

    for record in samples:
        generated, draft = generate_stylized(record)

        print("=" * 80)
        print(f"AUTHOR: {author_id}  |  passage: {record.passage_id}")
        print()
        print("--- DRAFT (input to model) ---")
        print(draft[:PRINT_CHARS])
        print()
        print("--- GENERATED REWRITE ---")
        print(generated[:PRINT_CHARS])
        print()
        print("--- REAL PASSAGE ---")
        print(clip_text(record.real_passage, PRINT_CHARS))
        print()